In [1]:
# 5. How does the historical success over the last 30-40 years of the 10 most popular active directors
# influence the financial and critical success of their new movie releases?

import os
import requests
import json
from datetime import datetime
from dotenv import load_dotenv
import time

load_dotenv('../../test.env')
api_key_TMDB = os.getenv("TMDB_API_KEY")

if not api_key_TMDB:
    print("API KEY not found check env file")
    exit()
else:
    print("KEY loaded")


KEY loaded


In [2]:
#Building our links for the different api
tmdb_base = "https://api.themoviedb.org/3"
tmbd_person_endpoint = "/person/popular"  #Get all result for "" fitting criteria
tmdb_url = tmdb_base + tmbd_person_endpoint

In [ ]:
# Calculate 10 years ago
current_year = datetime.now().year
start_year = current_year - 10
start_date = f"{start_year}-01-01"

#Final list of the 15 most popular directors
dir_result = []
total_pages = 1

parameters = {
    'api_key' : api_key_TMDB,
    'page' : 1
}
while len(dir_result) < 15 and parameters['page'] <= total_pages:
    person_response = requests.get(tmdb_url, params=parameters)
    
    if person_response.status_code == 200:
        person_data = person_response.json()
        total_pages = person_data['total_pages']

        # Looking for people known for directing
        for person in person_data['results']:

            # Checking if Actor or Director
            if person['known_for_department'] == 'Directing':
                person_id = person['id']
                # request link to get details about each director, to check if still active
                detail_url = f"https://api.themoviedb.org/3/person/{person['id']}/movie_credits"
                detail_params = {
                    'api_key' : api_key_TMDB
                }
                
                try:
                    detail_response = requests.get(detail_url, params=parameters)
                    if detail_response.status_code == 200:
                        detail_data = detail_response.json()
                except:
                    pass
                
                for movie in detail_data['crew']:
                    if movie['job'] == 'Director' and movie['release_date'] >= start_date:
                        dir_result.append({
                            'name' : person['name'],
                            'tmdb_id' : person['id']
                        })
                    break


        parameters['page']+=1

        #Spam protection
        time.sleep(0.25)
                #liste von 50 directorn machen und diese nach popularity auf 15 filtern?
                #ansonsten fuer andere frage schauen, wieviele gute filme die gemacht haben und, wie sich das ueber die zeit entwickelt hat

    else:
        print(f"Error on page {parameters['page']}")
        break


print(dir_result)

In [ ]:
#list of all movies of the directors for the last 35 years
dir_movie_list = []
total_pages = 1


for person in dir_result:

    person_id=person['tmdb_id']
    director_url = f"https://api.themoviedb.org/3/person/{person_id}/movie_credits"
    parameters = {
        'api_key' : api_key_TMDB
    }

    try:
        director_response = requests.get(director_url,params=parameters)
        if director_response.status_code == 200:
            director_data = director_response.json()
    except:
        pass
    
    for movie in person['crew']:
        if movie['job'] == "Director":
            movie_id = movie['id']
            movie_url=f"https://api.themoviedb.org/3/movie/{movie_id}"
            parameters = {
                'api_key' : api_key_TMDB
            }

            try:
                movie_response = requests.get(movie_url, params=parameters)
                if movie_response.status_code == 200:
                    movie_data = movie_response.json()
            except:
                pass


            dir_movie_list.append({
                'dir_name' : person['name'],
                'dir_id' : person['tmdb_id'],
                'movie_title' : movie_data['title'],
                'movie_id' : movie_data['id'],
                'budget' : movie_data['budget'],
                'revenue' : movie_data['revenue'],
                'rating' : movie_data['vote_average'],
                'rating_count' : movie_data['vote_count'],
                'popularity' : movie_data['popularity']

            })
        
        time.sleep(0.25)

    time.sleep(0.25)